In [2]:
import wheel
import pandas as pd
import numpy as np
import sklearn as skl
from mat4py import loadmat
import scipy.io
import csv
import struct
import h5py

In [3]:
!pip3 install --upgrade wslink aiohttp


[notice] A new release of pip is available: 24.1.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import os 
os.environ['MKL_NUM_THREADS'] = '1' 
os.environ['NUMEXPR_NUM_THREADS'] = '1' 
os.environ['OMP_NUM_THREADS'] = '1'

In [5]:
from aiohttp import ClientTimeout
timeout = ClientTimeout(total=600)  
# Use this timeout when creating your client session

In [6]:
#Can minimize this cell is not needed
#MATLAB SCIPT used to generate converted mat files compatible with python
"""
% TRIGGER_LIST_TO_CELL_STRUCT
iter = length(trig_list.sentence); 
trigg_list = table2struct(trig_list);
struct(trigg_list); 

for i = 1:iter
   % Check if the cell contains a table
   if istable(trigg_list(i).sentence)
       % Get the table
       tableData = trigg_list(i).sentence;
     
       % Iterate through each variable (column) in the table
       varNames = tableData.Properties.VariableNames;
       for j = 1:numel(varNames)
           % Get the column data
           columnData = tableData.(varNames{j});
         
           % Check if the column data contains strings and convert to cell array
           if isstring(columnData)
               % Convert string column data to cell array of strings
               tableData.(varNames{j}) = cellstr(columnData);
           end
       end
     
       % Replace the updated table back into the cell array
       trigg_list(i).sentence = tableData;
   end
end
%%Step 2
% Assuming curr_state.trigger_list is a struct array
sen_field = cell(iter, 1);  % Preallocate cell array for output
emptyMap = containers.Map();  % Create an empty dictionary
for j = 1:iter
   % Check if 'sentence' field of the current struct entry is an empty list []
   if not(istable(trigg_list(j).sentence))
       % If sentence is empty, assign empty dictionary to S{i}
       sen_field{j} = emptyMap;
   else
       % Convert the 'sentence' field of the current struct entry to a struct
       % Assuming curr_state.trigger_list(i).sentence is convertible to a table or struct
       if istable(trigg_list(j).sentence)
           sen_field{j} = table2struct(trigg_list(j).sentence);
           struct(sen_field{j})
       else
           % Handle case where sentence is not a table but some other structure
           % Adjust conversion or handling as needed
           sen_field{j} = emptyMap;
       end
   end
end
disp(j)
sentence_out = cell2table(sen_field);
sentence_work = table2struct(sentence_out);
"""

"\n% TRIGGER_LIST_TO_CELL_STRUCT\niter = length(trig_list.sentence); \ntrigg_list = table2struct(trig_list);\nstruct(trigg_list); \n\nfor i = 1:iter\n   % Check if the cell contains a table\n   if istable(trigg_list(i).sentence)\n       % Get the table\n       tableData = trigg_list(i).sentence;\n     \n       % Iterate through each variable (column) in the table\n       varNames = tableData.Properties.VariableNames;\n       for j = 1:numel(varNames)\n           % Get the column data\n           columnData = tableData.(varNames{j});\n         \n           % Check if the column data contains strings and convert to cell array\n           if isstring(columnData)\n               % Convert string column data to cell array of strings\n               tableData.(varNames{j}) = cellstr(columnData);\n           end\n       end\n     \n       % Replace the updated table back into the cell array\n       trigg_list(i).sentence = tableData;\n   end\nend\n%%Step 2\n% Assuming curr_state.trigger_list 

In [7]:
#MATLAB SCRIPT
"""
%CURRSTATE TO STRUCT

curr_state_export = struct(curr_state); 
curr_state_export.filenames = struct(curr_state_export.filenames);
curr_state_export.presentation_matrix = struct(curr_state_export.presentation_matrix);

"""

'\n%CURRSTATE TO STRUCT\n\ncurr_state_export = struct(curr_state); \ncurr_state_export.filenames = struct(curr_state_export.filenames);\ncurr_state_export.presentation_matrix = struct(curr_state_export.presentation_matrix);\n\n'

In [8]:
"""
To use to notebook the following files are needed (So far files are only generated for subject 8 and 14 but more files can be created
using matlab scripts) 
-elecFinal table
-Python friendly converted tables (tables and strings are converted to cell arrays, every is converted to structs to avoid opaque issues)
-HDR, CurrState, Record, Trig-List raw, Trig-list sentence field
First module sets up the task data, second part visualizes electrodes, can comment everything above the h5py line if only neurological 
processing is needed
-Also change the variable Gen_Path to your local directory, Subject to the subject you want and Sub_Num to that subjects number
"""
#neurological dependencias 
#REMEMBER TO RUN THIS CELL!
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colormaps
from mne_bids import BIDSPath, read_raw_bids
from scipy.signal import butter, filtfilt
from mne.time_frequency import tfr_multitaper
import seaborn as sb
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
import tkinter as tk



import mne
from mne.viz import plot_alignment, snapshot_brain_montage

In [9]:
import mne_bids
from mne.channels import make_standard_montage

In [10]:
#Trouble shooting
%matplotlib inline

In [11]:
#TODO: Think about wrapping some of these "Modules" into functions
#issue with 17 
#Might have to change some keys <- may be some consistency issues

In [12]:
#Import in Subject Files
#GLOBAL VARS CHANGE FOR INTENDED SUBJECT
Subject = "S13"; #Change the number after S to choose another subject (Options are 1 through 17)
Sub_num = 13;
#Replace this path with where subject converted files are found on your local
Gen_Path = "/Volumes/Expansion/4WT/COLLAB-CODE/Alliyah/Sub-Mat-Converted/"+Subject+"/"
record_direc = "s13_ccf_SANN006" #Look at the original directory names to change the variable accordingly 

In [13]:
#STEP 1: Handeling Behavioral Data
#Defining Paths for Files to Be Imported 
Curr_State = Gen_Path+Subject+"-CurrState.mat"
Trig_Sentence = Gen_Path+Subject+"-sentence.mat"
Trigger = Gen_Path+Subject+"-trig.mat"

In [14]:
print(Gen_Path)
print(Curr_State)
print(Trig_Sentence)
print(Trigger)

/Volumes/Expansion/4WT/COLLAB-CODE/Alliyah/Sub-Mat-Converted/S13/
/Volumes/Expansion/4WT/COLLAB-CODE/Alliyah/Sub-Mat-Converted/S13/S13-CurrState.mat
/Volumes/Expansion/4WT/COLLAB-CODE/Alliyah/Sub-Mat-Converted/S13/S13-sentence.mat
/Volumes/Expansion/4WT/COLLAB-CODE/Alliyah/Sub-Mat-Converted/S13/S13-trig.mat


In [15]:
#Loading in neccesary raw files
curr_raw = scipy.io.loadmat(Curr_State,squeeze_me=True, struct_as_record = False, simplify_cells = True)
trig_raw = scipy.io.loadmat(Trigger, squeeze_me=True, struct_as_record = False, simplify_cells = True)
sentence_raw = scipy.io.loadmat(Trig_Sentence, squeeze_me=True, struct_as_record = False, simplify_cells = True)

In [16]:
trig_raw.keys()

dict_keys(['__header__', '__version__', '__globals__', 'trigg_list', '__function_workspace__'])

In [17]:
#Creating seperate data frames of the nested struct fields
presentation_matrix_frame = pd.DataFrame(curr_raw['curr_state_export']['presentation_matrix'])
trigger_frame = pd.DataFrame(trig_raw['trigg_list'])
file_frame =pd.DataFrame(list(curr_raw['curr_state_export']['filenames']))

In [18]:
#Slicing off the Curr State Table
curr_state_slice = curr_raw['curr_state_export']

In [19]:
#Create seperate dictionary of all columns that can be resolved to just a constant
constants = list(curr_state_slice.keys())
#remove the class type fields
constants.remove('trigger_list')
constants.remove('presentation_matrix')
constants.remove('blockList')
constants.remove('filenames')
#Use dictionary comprehension to only keep primitive type data in final dictionary
constants_frame = {key: curr_state_slice[key] for key in constants}

In [20]:
len(sentence_raw['sentence_work'])

8313

In [21]:
#sentence_raw['sentence_work']

In [22]:
#replacing MATLAB opaque sentences with properly parsed ones
sentence_dict = sentence_raw['sentence_work']
trigger_frame['sentence'] = sentence_dict


In [23]:
#STEP 2: Handle the neurological data

In [24]:
#define paths for record and hdr
hdr_path = Gen_Path+Subject+"-hdr.mat"

In [25]:
#Load in hdr
hdr_raw = curr_raw = scipy.io.loadmat(hdr_path,squeeze_me=True, struct_as_record = False, simplify_cells = True)
prim_keys = ['ver', 'patientID', 'recordID', 'startdate', 'starttime', 'bytes', 'records', 'duration', 'ns', 'label', 'labelNew']
hdr_class_dict = {key: value for key, value in hdr_raw['hdr'].items() if key not in prim_keys}
hdr_prim_dict= {key: value for key, value in hdr_raw['hdr'].items() if key in prim_keys}

In [26]:
hdr_class_dict.keys()

dict_keys(['labelOld', 'transducer', 'units', 'physicalMin', 'physicalMax', 'digitalMin', 'digitalMax', 'prefilter', 'samples', 'frequency'])

In [27]:
hdr_frame = pd.DataFrame(hdr_class_dict)

In [28]:
hdr_class_dict.keys()

dict_keys(['labelOld', 'transducer', 'units', 'physicalMin', 'physicalMax', 'digitalMin', 'digitalMax', 'prefilter', 'samples', 'frequency'])

In [29]:
class_ex_label = pd.DataFrame(hdr_class_dict)

In [30]:
len(hdr_raw['hdr']['label'])


175

In [31]:
labels = list(hdr_raw['hdr']['label'])

In [32]:
labels

['A1',
 'A2',
 'A3',
 'A4',
 'A5',
 'A6',
 'A7',
 'A8',
 'A9',
 'A10',
 'B1',
 'B2',
 'B3',
 'B4',
 'B5',
 'B6',
 'B7',
 'B8',
 'B9',
 'B10',
 'Ci1',
 'Ci2',
 'Ci3',
 'Ci4',
 'Ci5',
 'Ci6',
 'Ci7',
 'Ci8',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'E1',
 'E2',
 'E3',
 'E4',
 'E5',
 'E6',
 'E7',
 'E8',
 'F1',
 'F2',
 'F3',
 'F4',
 'F5',
 'F6',
 'F7',
 'F8',
 'F9',
 'F10',
 'K1',
 'K2',
 'K3',
 'K4',
 'K5',
 'K6',
 'K7',
 'K8',
 'K9',
 'K10',
 'K11',
 'K12',
 'K13',
 'K14',
 'L1',
 'L2',
 'L3',
 'L4',
 'L5',
 'L6',
 'L7',
 'L8',
 'L9',
 'L10',
 'O1',
 'O2',
 'O3',
 'O4',
 'O5',
 'O6',
 'O7',
 'O8',
 'P1',
 'P2',
 'P3',
 'P4',
 'P5',
 'P6',
 'P7',
 'P8',
 'S1',
 'S2',
 'S3',
 'S4',
 'S5',
 'S6',
 'S7',
 'S8',
 'T1',
 'T2',
 'T3',
 'T4',
 'U1',
 'U2',
 'U3',
 'U4',
 'U5',
 'U6',
 'U7',
 'U8',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'W1',
 'W2',
 'W3',
 'W4',
 'W5',
 'W6',
 'W7',
 'W8',
 'W9',
 'W10',
 'W11',
 'W1

In [33]:
#Use original data path for this part 
#Replace this path each time for different subjects
record_path = "/Volumes/Expansion/4WT/4WT-analysis/DATA/mainfiles/"+record_direc+"/neuralMatfile/FILE1.mat"

In [34]:
with h5py.File(record_path, 'a') as f: #a means both read and write access is permited
    data = f['record'][:] #change this to record instead of data for specificity 

In [35]:
record_frame = pd.DataFrame(data)

In [36]:
trigger_frame['sentence'][4].keys()

dict_keys(['sen_field'])

In [37]:
trigger_frame.iloc[0]

Type                                                                  START
number                                                                    8
trig_sent                                                                 1
system_timePreOnset                                              720.894669
system_timePostTrigger                                           721.627337
sentence                  {'sen_field': [[b'', b'MCOS', b'containers.Map...
block_num                                                                -1
stim_num                                                                 -1
response                                                                 -1
Var1                                                                  44441
Var2                                                                  44441
Name: 0, dtype: object

In [38]:
trigger_frame.columns

Index(['Type', 'number', 'trig_sent', 'system_timePreOnset',
       'system_timePostTrigger', 'sentence', 'block_num', 'stim_num',
       'response', 'Var1', 'Var2'],
      dtype='object')

In [39]:
trigger_frame['Type'].value_counts()

Type
SW_FIXN               894
SW_WORD               782
FIXATION1_AUD         455
WAIT_BEFORE_Q_AUD     454
IMAGE_QUESTION_AUD    454
WORD3_AUD             454
WORD2_AUD             454
WORD1_AUD             454
WORD4_AUD             454
FIXATION1_VIS         453
IMAGE_QUESTION_VIS    452
WAIT_BEFORE_Q_VIS     452
WORD4_VIS             452
WORD3_VIS             452
WORD2_VIS             452
WORD1_VIS             452
SW_HASH               120
GIF                    78
SCORE                  42
SW_START               16
Testing                13
SW_FIX1                 8
NEW_BLOCK               7
PAUSE_BLOCK             5
START                   2
ALL_BLOCKS_END          1
END                     1
Name: count, dtype: int64

In [40]:
trigger_frame.iloc[0]['Type']

'START'

In [41]:
trigger_frame['sentence'].value_counts()

sentence
{'sen_field': {'word': '########', 'type': 'CCC'}}                                                                         240
{'sen_field': {'word': 'girls', 'type': 'ANI'}}                                                                             84
{'sen_field': {'word': 'boys', 'type': 'ANI'}}                                                                              64
{'sen_field': {'word': 'kids', 'type': 'ANI'}}                                                                              56
{'sen_field': {'word': 'men', 'type': 'ANI'}}                                                                               48
                                                                                                                          ... 
{'sen_field': [[b'', b'MCOS', b'containers.Map', [3707764736          2          1          1          1          1]]]}      1
{'sen_field': [[b'', b'MCOS', b'containers.Map', [3707764736          2          1          1         

In [42]:
empty_dict = {
 'w1': '',
 'w2': '',
 'w3': '',
 'w4': '',
 'EN_translation': '',
 'imageFile': '',
 'falseImageFile': '',
 'relatedImage': '',
 'sentenceType': '',
 'w1Type': '',
 'w2Type': '',
 'w3Type': '',
 'w4Type': '',
 'w1SoundFile': '',
 'w2SoundFile': '',
 'w3SoundFile': '',
 'w4SoundFile': '',
 'modality': ''
}

In [43]:
#un-nesting the dictionaries
num = len(trigger_frame['sentence'])
for i in range(num):
    trigger_frame.at[i, 'sentence'] = trigger_frame.at[i, 'sentence']['sen_field']

In [ ]:
"""
{'w1': 'the',
 'w2': 'women',
 'w3': 'wrote',
 'w4': 'poems',
 'imageFile': '/Users/klab/Desktop/4WT-English/word-sentence-lists/sentence-pics/the-women-wrote-poems.jpg',
 'falseImageFile': '/Users/klab/Desktop/4WT-English/word-sentence-lists/sentence-pics/the-girls-rode-bikes.jpg',
 'relatedImage': 1,
 'sentenceType': 'GS',
 'w1Type': 'det',
 'w2Type': 'nna',
 'w3Type': 'ver',
 'w4Type': 'nno',
 'modality': 'a'}
"""

In [81]:
#Getting rid of Matlab empty maps within the raw data
count = 0
for i in range(num):
    if not isinstance(trigger_frame.loc[i, 'sentence'], dict):
        trigger_frame.at[i, 'sentence'] = empty_dict
        count+=1
    if 'word' in trigger_frame.loc[i, 'sentence'] or 'type' in trigger_frame.loc[i, 'sentence']:
        trigger_frame.drop(i)
        

In [ ]:
corr_map = {'W1':'w1',
            'W2':'w2',
            'W3':'w3',
            'W4':'w4',
            'word1type':'w1Type',
            'word2type':'w2Type',
            'word3type':'w3Type',
            'word4type':'w4Type',
            'falseimageFile': 'falseImageFile',
           }

In [88]:
sen_list[200].keys()

dict_keys(['w1', 'w2', 'w3', 'w4', 'imageFile', 'falseImageFile', 'relatedImage', 'sentenceType', 'w1Type', 'w2Type', 'w3Type', 'w4Type', 'modality'])

In [45]:
count

136

In [46]:
sen_list = list(trigger_frame['sentence'])

In [82]:
#replace incorrect keys 
for i,item in enumerate(sen_list):
    for key in corr_map:
        if key in item:
            sen_list[i][corr_map[key]] = sen_list[i].pop(key)

In [100]:
sen_frame = pd.DataFrame(sen_list, columns = ['w1', 'w2', 'w3', 'w4', 'imageFile', 'falseImageFile', 'relatedImage', 'sentenceType', 'w1Type', 'w2Type', 'w3Type', 'w4Type', 'modality'])

In [102]:
sen_frame.dropna(inplace = True, how = 'all')

In [103]:
sen_frame

,w1,w2,w3,w4,imageFile,falseImageFile,relatedImage,sentenceType,w1Type,w2Type,w3Type,w4Type,modality
0,,,,,,,,,,,,,
1,the,wrote,women,poems,/Users/klab/Desktop/4WT-English/word-sentence-...,/Users/klab/Desktop/4WT-English/word-sentence-...,1,NGNS,det,ver,nna,nno,v
2,the,wrote,women,poems,/Users/klab/Desktop/4WT-English/word-sentence-...,/Users/klab/Desktop/4WT-English/word-sentence-...,1,NGNS,det,ver,nna,nno,v
3,the,wrote,women,poems,/Users/klab/Desktop/4WT-English/word-sentence-...,/Users/klab/Desktop/4WT-English/word-sentence-...,1,NGNS,det,ver,nna,nno,v
4,the,wrote,women,poems,/Users/klab/Desktop/4WT-English/word-sentence-...,/Users/klab/Desktop/4WT-English/word-sentence-...,1,NGNS,det,ver,nna,nno,v
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8072,the,stole,monkeys,mangoes,/Users/klab/Desktop/4WT-English/word-sentence-...,/Users/klab/Desktop/4WT-English/word-sentence-...,0,NGNS,det,ver,nna,nno,a
8073,NA,NA,NA,NA,NA,NA,-1,NA,NA,NA,NA,NA,NA
8074,NA,NA,NA,NA,NA,NA,-1,NA,NA,NA,NA,NA,NA
8311,,,,,,,,,,,,,


In [98]:
sen_frame.value_counts()

w1   w2       w3       w4      imageFile                                                                                        falseImageFile                                                                                   relatedImage  sentenceType  w1Type  w2Type  w3Type  w4Type  modality
                                                                                                                                                                                                                                                                                                         136
NA   NA       NA       NA      NA                                                                                               NA                                                                                               -1            NA            NA      NA      NA      NA      NA           29
the  laid     ducks    eggs    /Users/klab/Desktop/4WT-English/word-sentence-lists/sentence-pics/the-duc

In [85]:
sen_frame.columns

Index(['w1', 'w2', 'w3', 'w4', 'EN_translation', 'imageFile', 'falseImageFile',
       'relatedImage', 'sentenceType', 'w1Type', 'w2Type', 'w3Type', 'w4Type',
       'w1SoundFile', 'w2SoundFile', 'w3SoundFile', 'w4SoundFile', 'modality',
       'word', 'type'],
      dtype='object')

In [50]:
trigger_frame['sentence'][452]

{'w1': 'the',
 'w2': 'kids',
 'w3': 'broke',
 'w4': 'plates',
 'imageFile': '/Users/klab/Desktop/4WT-English/word-sentence-lists/sentence-pics/the-kids-broke-plates.jpg',
 'falseImageFile': '/Users/klab/Desktop/4WT-English/word-sentence-lists/sentence-pics/the-sisters-boarded-planes.jpg',
 'relatedImage': 1,
 'sentenceType': 'GS',
 'w1Type': 'det',
 'w2Type': 'nna',
 'w3Type': 'ver',
 'w4Type': 'nno',
 'modality': 'a'}

In [51]:
trigger_frame

,Type,number,trig_sent,system_timePreOnset,system_timePostTrigger,sentence,block_num,stim_num,response,Var1,Var2
0,START,8,1,720.894669,721.627337,"{'w1': '', 'w2': '', 'w3': '', 'w4': '', 'EN_t...",-1,-1,-1,44441,44441
1,FIXATION1_VIS,1,1,721.646278,721.677475,"{'w1': 'the', 'w2': 'wrote', 'w3': 'women', 'w...",1,1,-1,45192,45192
2,WORD1_VIS,1,1,722.255592,722.292572,"{'w1': 'the', 'w2': 'wrote', 'w3': 'women', 'w...",1,1,-1,45808,45808
3,WORD2_VIS,1,1,723.133988,723.159476,"{'w1': 'the', 'w2': 'wrote', 'w3': 'women', 'w...",1,1,-1,46675,46675
4,WORD3_VIS,1,1,724.011917,724.042228,"{'w1': 'the', 'w2': 'wrote', 'w3': 'women', 'w...",1,1,-1,47558,47558
...,...,...,...,...,...,...,...,...,...,...,...
8308,SW_HASH,1,1,4918.940846,4918.970843,"{'word': '########', 'type': 'CCC'}",8,117,"{'trial_success': -4, 'buttonpressed': -1, 're...",11039035,1
8309,SW_FIXN,1,1,4919.797948,4919.819712,"{'word': '########', 'type': 'CCC'}",8,118,-1,11039885,1
8310,SW_HASH,1,1,4920.403250,4920.436508,"{'word': '########', 'type': 'CCC'}",8,118,"{'trial_success': 1, 'buttonpressed': 1, 'resp...",11040502,1
8311,ALL_BLOCKS_END,5,1,4921.266296,4921.705210,"{'w1': '', 'w2': '', 'w3': '', 'w4': '', 'EN_t...",9,-1,-1,11041368,5


In [52]:
#vis_w2ver = arr_find_in_sentence_v2(trigger_frame, 2, "ver", 2)
def arr_find_in_sentence_v2(trigger_frame, ev_num, w_type, modality):
    # Take number of rows in the trigger list
    n_events = len(trigger_frame)
    # Create a binary mapping that starts with every value being false
    sel = np.zeros(n_events, dtype=bool)
    # Looping through all events
    for i in range(n_events):
        #Take the current row 
        event = trigger_frame.iloc[i].to_dict()
        #pull off the sentence dictionary
        sentence = event['sentence']
        #grab the event type
        event_type = event['Type']
        #Go into a switch case like logical statement to determine how to set the mapping
        try:
            if sentence != {} and ('w1Type' in sentence or 'relatedImage' in sentence):
                match = False
                if ev_num == 1 and 'w1Type' in sentence:
                    match = sentence['w1Type'] == w_type and 'WORD1' in event_type
                elif ev_num == 2 and 'w2Type' in sentence:
                    match = sentence['w2Type'] == w_type and 'WORD2' in event_type
                elif ev_num == 3 and 'w3Type' in sentence:
                    match = sentence['w3Type'] == w_type and 'WORD3' in event_type
                elif ev_num == 4 and 'w4Type' in sentence:
                    match = sentence['w4Type'] == w_type and 'WORD4' in event_type
                elif ev_num == 5 and 'sentenceType' in sentence:
                    match = sentence['sentenceType'] == w_type and 'WAIT' in event_type
                elif ev_num == 6 and 'relatedImage' in sentence:
                    match = sentence['relatedImage'] == w_type and 'QUESTION' in event_type
                elif ev_num == 7 and 'Type' in sentence:
                    match = sentence['Type'] == w_type and 'SW_WORD' in event_type
                elif ev_num == 8 and 'SW_HASH' in event_type:
                    match = True

                if match:
                    sel[i] = True
        except KeyError:
            pass
    
    #In the case of an ev_num less than 7 take into count audio or visual within the type field
    if ev_num < 7:
        aud_sel = np.array(['_AUD' in event['Type'] for event in trigger_frame.to_dict('records')])
        vis_sel = np.array(['_VIS' in event['Type'] for event in trigger_frame.to_dict('records')])
        #In modality audio is a and visual is v
        if modality == 1:
            sel = sel & aud_sel
        else:
            sel = sel & vis_sel
   
    return sel.tolist()

In [53]:
#uncomment this code and choose a variable below to see True falses in the mapping and to test functionality
"""
unique, counts = np.unique(aud_w2noun, return_counts = True)
dict(zip(unique, counts)) #function is working properly
"""

'\nunique, counts = np.unique(aud_w2noun, return_counts = True)\ndict(zip(unique, counts)) #function is working properly\n'

In [54]:
trigger_frame['sentence'][500]

{'word': 'wine', 'type': 'UNA'}

In [55]:
aud_w2nna = arr_find_in_sentence_v2(trigger_frame, 2, "nna", 1)
aud_w3nna = arr_find_in_sentence_v2(trigger_frame, 3, "nna", 1)
aud_w4nna = arr_find_in_sentence_v2(trigger_frame, 4, "nna", 1)

In [56]:
aud_w2nno = arr_find_in_sentence_v2(trigger_frame, 2, "nno", 1)
aud_w3nno = arr_find_in_sentence_v2(trigger_frame, 3, "nno", 1)
aud_w4nno = arr_find_in_sentence_v2(trigger_frame, 4, "nno", 1)

In [57]:
aud_w2ver = arr_find_in_sentence_v2(trigger_frame, 2, "ver", 1)
aud_w3ver = arr_find_in_sentence_v2(trigger_frame, 3, "ver", 1)
aud_w4ver = arr_find_in_sentence_v2(trigger_frame, 4, "ver", 1)

In [58]:
aud_wwS1 = arr_find_in_sentence_v2(trigger_frame, 5, "GS", 1)
aud_wwS2 = arr_find_in_sentence_v2(trigger_frame, 5, "GNS", 1)
aud_wwS3 = arr_find_in_sentence_v2(trigger_frame, 5, "NGNS", 1)

In [59]:
aud_wwIR = arr_find_in_sentence_v2(trigger_frame, 6, 1, 1)
aud_wwIN = arr_find_in_sentence_v2(trigger_frame, 6, 0, 1)

In [60]:
vis_w2nna = arr_find_in_sentence_v2(trigger_frame, 2, "nna", 2)
vis_w3nna = arr_find_in_sentence_v2(trigger_frame, 3, "nna", 2)
vis_w4nna = arr_find_in_sentence_v2(trigger_frame, 4, "nna", 2)

In [61]:
vis_w2nno = arr_find_in_sentence_v2(trigger_frame, 2, "nno", 2)
vis_w3nno = arr_find_in_sentence_v2(trigger_frame, 3, "nno", 2)
vis_w4nno = arr_find_in_sentence_v2(trigger_frame, 4, "nno", 2)

In [62]:
vis_w2ver = arr_find_in_sentence_v2(trigger_frame, 2, "ver", 2)
vis_w3ver = arr_find_in_sentence_v2(trigger_frame, 3, "ver", 2)
vis_w4ver = arr_find_in_sentence_v2(trigger_frame, 4, "ver", 2)

In [63]:
vis_wwS1 = arr_find_in_sentence_v2(trigger_frame, 5, "GS", 2)
vis_wwS2 = arr_find_in_sentence_v2(trigger_frame, 5, "GNS", 2)
vis_wwS3 = arr_find_in_sentence_v2(trigger_frame, 5, "NGNS", 2)

In [64]:
vis_wwIR = arr_find_in_sentence_v2(trigger_frame, 6, 1, 2)
vis_wwIN = arr_find_in_sentence_v2(trigger_frame, 6, 0, 2)

In [65]:
#Main Comparisons
aud_w2noun = [a or b for a, b in zip(aud_w2nno, aud_w2nna)]
aud_w3noun = [a or b for a, b in zip(aud_w3nno, aud_w3nna)]
aud_w4noun = [a or b for a, b in zip(aud_w4nno, aud_w4nna)]
vis_w2noun = [a or b for a, b in zip(vis_w2nno, vis_w2nna)]
vis_w3noun = [a or b for a, b in zip(vis_w3nno, vis_w3nna)]
vis_w4noun = [a or b for a, b in zip(vis_w4nno, vis_w4nna)]

ctrl_ANI = arr_find_in_sentence_v2(trigger_frame, 7, 'ANI', 2)
ctrl_UNA = arr_find_in_sentence_v2(trigger_frame, 7, 'UNA', 2)
ctrl_VER = arr_find_in_sentence_v2(trigger_frame, 7, 'VERB', 2)
ctrl_HSH = arr_find_in_sentence_v2(trigger_frame, 8, 'HASH', 2)

In [66]:
def notch_filter(signal, fs, freq=60.0):
    Q = 30.0  # Quality factor
    b, a = butter(2, [freq - 1, freq + 1], btype='bandstop', fs=fs)
    return filtfilt(b, a, signal)

In [67]:
tapers = [3, 4]
params_Hgamma = {
    'fpass': [30, 150],  # Frequency range
    'tapers': tapers,
    'Fs': hdr_frame['frequency'][0], # Sampling frequency
    'trialave': 0
}

In [68]:
# Assuming the existence of the required variables like record_frame, hdr_frame, trigger_frame, labels, etc.
record = record_frame.transpose()
nElecs = len(labels)
fs = hdr_frame.frequency[0]

timeFIXBSL = -400
timeWord = 1100
idxFIXBSL = int(np.ceil(timeFIXBSL * fs / 1000))
idxWord = int(np.ceil(timeWord * fs / 1000))
sizeTime = idxWord - idxFIXBSL

# Butterworth filter coefficients
b, a = butter(5, 200 / (fs / 2), btype='low')

# Ensure correct indexing for triggers
idxFX_S1 = np.where(aud_wwS1)[0] - 5 + 1
idxFX_S2 = np.where(aud_wwS2)[0] - 5 + 1
idxFX_S3 = np.where(aud_wwS3)[0] - 5 + 1

# Initialize an empty DataFrame for filtered values
filtered_data = np.zeros((nElecs, sizeTime))
for elecNum in range(nElecs - 1):
    try:
        # Perform bipolar montage
        elecSignal = record.iloc[elecNum, :] - record.iloc[elecNum + 1, :]
        
        # Apply notch filter and low-pass filter
        elecSignal = filtfilt(b, a, notch_filter(elecSignal, fs))

        # Synchronize with triggers
        data_S1 = np.array([elecSignal[trigger_frame.iloc[idx]['Var1'] + idxFIXBSL:trigger_frame.iloc[idx]['Var1'] + idxWord] 
                            for idx in idxFX_S1 if trigger_frame.iloc[idx]['Var1'] + idxWord < len(elecSignal)])
        data_S2 = np.array([elecSignal[trigger_frame.iloc[idx]['Var1'] + idxFIXBSL:trigger_frame.iloc[idx]['Var1'] + idxWord] 
                            for idx in idxFX_S2 if trigger_frame.iloc[idx]['Var1'] + idxWord < len(elecSignal)])
        data_S3 = np.array([elecSignal[trigger_frame.iloc[idx]['Var1'] + idxFIXBSL:trigger_frame.iloc[idx]['Var1'] + idxWord] 
                            for idx in idxFX_S3 if trigger_frame.iloc[idx]['Var1'] + idxWord < len(elecSignal)])

        # Store filtered data in the DataFrame
        filtered_data[elecNum, :] = np.mean(data_S1, axis=0)  # Example: Use mean value for simplicity

    except Exception as e:
        print(f"Error processing electrode {elecNum + 1}: {e}")

# Create a new DataFrame with filtered data
filtered_df = pd.DataFrame(filtered_data, columns=np.arange(sizeTime))




In [69]:
filtered_df

,0,1,2,3,4,5,6,7,8,9,...,1490,1491,1492,1493,1494,1495,1496,1497,1498,1499
0,4.083010,3.839207,3.445251,3.003543,2.675605,2.564632,2.619377,2.683000,2.646254,2.537318,...,-3.588230,-3.294095,-2.950622,-2.666655,-2.517510,-2.457582,-2.355111,-2.130824,-1.845891,-1.637221
1,-1.735731,-1.799838,-1.810824,-1.777883,-1.726029,-1.687620,-1.668677,-1.635293,-1.556562,-1.462516,...,-10.556462,-10.036565,-9.634105,-9.463866,-9.537966,-9.781979,-10.096065,-10.400083,-10.639602,-10.772228
2,31.281350,31.314740,31.403480,31.526378,31.635807,31.692792,31.692992,31.656238,31.598956,31.528313,...,22.049415,21.808288,21.694464,21.767787,22.031736,22.407701,22.750752,22.927214,22.899337,22.731161
3,-30.961021,-30.954837,-30.916872,-30.858734,-30.806159,-30.779797,-30.783605,-30.813190,-30.865900,-30.933901,...,-34.271238,-34.431331,-34.560703,-34.633703,-34.651367,-34.633223,-34.603458,-34.578786,-34.564070,-34.557917
4,8.268038,8.297789,8.321447,8.305348,8.257959,8.212464,8.189084,8.183076,8.180674,8.171690,...,-0.432827,-0.622630,-0.813306,-0.979225,-1.124762,-1.265549,-1.397247,-1.497215,-1.559608,-1.613176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,240.124745,234.280316,230.214587,232.689976,240.565441,249.244845,253.799480,250.618902,239.218250,224.108151,...,129.353069,116.623432,121.321972,141.122749,169.083928,199.411645,229.555152,257.914589,281.748032,298.232100
171,105062.128654,101886.308694,98564.408667,97261.517131,98106.947122,99601.623941,100401.261334,100395.444491,100258.171261,100423.035448,...,93558.138095,93053.488674,92306.628228,91111.954481,89509.302816,87929.667471,86961.310108,86924.738271,87647.890089,88563.259731
172,-35.927397,-33.499326,-31.197522,-31.326432,-33.519113,-35.209935,-34.548446,-32.398200,-31.197400,-32.093531,...,-32.062846,-31.080535,-31.059346,-31.282488,-30.749160,-29.462554,-28.573497,-29.124461,-30.739154,-31.844432
173,30.246313,29.697498,29.020351,28.818084,29.150202,29.512477,29.431191,28.976924,28.639848,28.751733,...,25.206295,24.973011,24.910867,24.907384,24.748684,24.391358,24.063476,24.050404,24.363819,24.680652


In [70]:
!pip3 install statsmodels


[notice] A new release of pip is available: 24.1.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip


In [71]:

import tkinter as tk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind
from statsmodels.stats.power import TTestIndPower

# Assuming `filtered_df` contains your filtered data
sfreq = 1000  # Sampling frequency in Hz
ch_names = ['Electrode{}'.format(i+1) for i in range(filtered_df.shape[0])]  # Channel names
info = mne.create_info(ch_names=ch_names, sfreq=sfreq)

# Convert DataFrame to numpy array
data = filtered_df.values  # Get the data values from the DataFrame

# Create Raw object
raw = mne.io.RawArray(data, info)

# Define periods
total_duration = 6600  # Total duration of the experiment in milliseconds
periods = {
    "Baseline": (0, 600),
    "W1": (600, 1475),
    "W2": (1475, 2350),
    "W3": (2350, 3225),
    "W4": (3225, 4100),
    "Consolidate": (4100, 5100),
    "Response": (5100, 6600)
}

# Find indices corresponding to baseline period
baseline_start_idx = int(periods["Baseline"][0] * sfreq / 1000)
baseline_end_idx = int(periods["Baseline"][1] * sfreq / 1000)

# Calculate baseline mean and standard deviation for each electrode
baseline_means = np.mean(data[:, baseline_start_idx:baseline_end_idx], axis=1)
baseline_stds = np.std(data[:, baseline_start_idx:baseline_end_idx], axis=1)

# Group electrodes into batches (10-15 electrodes per plot)
electrode_groups = [ch_names[i:i+15] for i in range(0, len(ch_names), 15)]

# Function to create a new window and plot
def plot_in_new_window(group, picks):
    root = tk.Tk()
    root.title(f'SEEG Data: {", ".join(group)}')
    
    fig, axs = plt.subplots(len(picks), 1, figsize=(15, 3*len(picks)), sharex=True)
    
    for i, pick in enumerate(picks):
        data, times = raw[pick, :]
        axs[i].plot(times, data.T, label=group[i], color=plt.cm.rainbow(i/len(picks)))
        
        # Mark periods
        for period, (start, end) in periods.items():
            start_prop = start / total_duration
            end_prop = end / total_duration
            axs[i].axvline(x=start_prop * times[-1], color='black', linestyle='--', linewidth=0.5)
            axs[i].axvline(x=end_prop * times[-1], color='black', linestyle='--', linewidth=0.5)
            axs[i].annotate(period, xy=((start_prop + end_prop) / 2 * times[-1], axs[i].get_ylim()[1]), 
                            xycoords='data', ha='center', va='bottom', 
                            bbox=dict(boxstyle="round,pad=0.3", fc="yellow", ec="black", lw=1))
        
        # Find and mark the maximum value in each word period if significant
        for period in ["W1", "W2", "W3", "W4"]:
            start, end = periods[period]
            start_prop = start / total_duration
            end_prop = end / total_duration
            start_idx = int(start_prop * len(times))
            end_idx = int(end_prop * len(times))
            period_data = data[:, start_idx:end_idx]
            if period_data.size > 0:  # Ensure the array is not empty
                max_val = np.max(period_data)
                max_time = times[start_idx:end_idx][np.argmax(period_data)]
                
                # Perform t-test between baseline and period data
                t_stat, p_val = ttest_ind(data[:, baseline_start_idx:baseline_end_idx].flatten(), period_data.flatten())
                
                # Annotate if p-value is significant (e.g., p < 0.05)
                if p_val < 0.05:
                    axs[i].plot(max_time, max_val, 'ro')
                    
                    # Compare maximum value with baseline statistics
                   
                    if max_val > baseline_means[pick] + 3 * baseline_stds[pick]:
                        axs[i].annotate(f'Significant\nMax: {max_val:.2f}', (max_time, max_val), 
                                        xytext=(5, 5), textcoords='offset points',
                                        bbox=dict(boxstyle="round,pad=0.3", fc="yellow", ec="black", lw=1))
               
        
        axs[i].set_ylabel(group[i])
        axs[i].legend()
    
    axs[-1].set_xlabel('Time (s)')
    axs[-1].set_xlim(0, times[-1])  # Set x-axis limit to the full duration of the data
    plt.tight_layout()
    
    canvas = FigureCanvasTkAgg(fig, master=root)
    canvas.draw()
    canvas.get_tk_widget().pack()
    
    root.mainloop()

# Plot each group of electrodes
"""
for group in electrode_groups:
    picks = [info.ch_names.index(ch_name) for ch_name in group]
    plot_in_new_window(group, picks)

"""



Creating RawArray with float64 data, n_channels=175, n_times=1500
    Range : 0 ... 1499 =      0.000 ...     1.499 secs
Ready.


'\nfor group in electrode_groups:\n    picks = [info.ch_names.index(ch_name) for ch_name in group]\n    plot_in_new_window(group, picks)\n\n'